<a href="https://colab.research.google.com/github/sispo3314/6th-BE-Blog/blob/main/Routing_Ablation_mHealth_0319.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Physics-Guided Patch-Level Time/Frequency Dual Transformer
# for mHealth — Routing Ablation Study
#
# Ablation variants:
#   (A) Fixed Average (0.5:0.5)
#   (B) Self-Attention Fusion (no physics)
#   (C) Physics-Guided Gate (Ours)
# ============================================================

import os
import math
import copy
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split

# ============================================================
# 0) Reproducibility / Device
# ============================================================
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# 1) Config
# ============================================================
ROOT = "/content/drive/MyDrive/datasets/MHEALTHDATASET"

BATCH_SIZE   = 128
EPOCHS       = 50
LR           = 1e-3
WEIGHT_DECAY = 5e-4
WARMUP_EP    = 5

WINDOW_SIZE  = 128           # 50Hz * 2.56s = 128
PATCH_SIZE   = 8
N_PATCHES    = WINDOW_SIZE // PATCH_SIZE
NUM_FREQ_BANDS = 8
STEP_SIZE    = 64            # 50% overlap

D_MODEL      = 64
NHEAD        = 4
NUM_LAYERS   = 2
D_FF         = 128
DROPOUT      = 0.1

GATE_HIDDEN        = 64
LABEL_SMOOTHING    = 0.1

TEST_SIZE    = 0.2
VAL_SIZE     = 0.1

# mHealth: 12 activities (label 0 = null/nothing, skip it)
ACTIVITY_NAMES = [
    "STANDING",       # 1
    "SITTING",        # 2
    "LYING",          # 3
    "WALKING",        # 4
    "STAIRS_UP",      # 5
    "WAIST_BEND",     # 6
    "FRONTAL_ELEV",   # 7
    "KNEES_BEND",     # 8
    "CYCLING",        # 9
    "JOGGING",        # 10
    "RUNNING",        # 11
    "JUMP_FRONT",     # 12
]
NUM_CLASSES = len(ACTIVITY_NAMES)  # 12

# Column indices (0-indexed) in the .log files (24 columns)
USE_COLS = [14, 15, 16, 17, 18, 19, 0, 1, 2]
LABEL_COL = 23
NUM_CHANNELS = 9

print(f"NUM_CLASSES : {NUM_CLASSES}")
print(f"NUM_CHANNELS: {NUM_CHANNELS}")
print(f"WINDOW_SIZE : {WINDOW_SIZE} (50Hz x 2.56s)")
print(f"TEST_SIZE   : {TEST_SIZE}")
print(f"VAL_SIZE    : {VAL_SIZE}")

# ============================================================
# 2) mHealth Loader + Train/Val/Test Split
# ============================================================
print("\nLoading mHealth...")

all_segments = []
all_labels = []

for subj_id in range(1, 11):  # 10 subjects
    fname = f"mHealth_subject{subj_id}.log"
    fpath = os.path.join(ROOT, fname)
    if not os.path.exists(fpath):
        print(f"  Skipping {fname} (not found)")
        continue

    data = np.loadtxt(fpath, dtype=np.float64)
    labels = data[:, LABEL_COL].astype(int)
    sensor = data[:, USE_COLS].astype(np.float32)

    # Handle NaN
    for c in range(sensor.shape[1]):
        col = sensor[:, c]
        nans = np.isnan(col)
        if nans.any():
            if nans.all():
                col[:] = 0.0
            else:
                not_nan = ~nans
                col[nans] = np.interp(
                    np.where(nans)[0], np.where(not_nan)[0], col[not_nan]
                )
            sensor[:, c] = col

    subj_count = 0
    T = len(sensor)

    for start in range(0, T - WINDOW_SIZE + 1, STEP_SIZE):
        end = start + WINDOW_SIZE
        window_labels = labels[start:end]

        if window_labels[0] != window_labels[-1]:
            continue
        unique = np.unique(window_labels)
        if len(unique) != 1:
            continue
        label = int(unique[0])

        if label == 0:
            continue
        label_mapped = label - 1

        window = sensor[start:end]
        if np.isnan(window).any():
            continue

        all_segments.append(window)
        all_labels.append(label_mapped)
        subj_count += 1

    print(f"  {fname}: {subj_count} windows")

X_all = np.array(all_segments, dtype=np.float32)
y_all = np.array(all_labels, dtype=np.int64)

print(f"\nTotal: {X_all.shape}, labels: {y_all.shape}")
print(f"Class distribution: {dict(zip(*np.unique(y_all, return_counts=True)))}")

# ── Step 1: Train+Val / Test (80/20, stratified) ──
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_all, y_all,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y_all,
)

# ── Step 2: Train+Val → Train / Val (90/10, stratified) ──
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=y_trainval,
)

print(f"\n{'='*50}")
print(f"Train : {X_train.shape}  ({len(y_train)} samples)")
print(f"Val   : {X_val.shape}  ({len(y_val)} samples)")
print(f"Test  : {X_test.shape}  ({len(y_test)} samples)")
print(f"{'='*50}")

# ── Normalization (train set 통계만 사용) ──
mu  = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True) + 1e-8
X_train = (X_train - mu) / std
X_val   = (X_val   - mu) / std
X_test  = (X_test  - mu) / std

print(f"\nNormalization done (train stats only)")
print(f"Train class dist: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Val   class dist: {dict(zip(*np.unique(y_val,   return_counts=True)))}")
print(f"Test  class dist: {dict(zip(*np.unique(y_test,  return_counts=True)))}")

# ============================================================
# 3) Physics Descriptor Utilities
# ============================================================
def spectral_entropy_1d(x, eps=1e-8):
    fft_mag = np.abs(np.fft.rfft(x))
    psd = fft_mag ** 2
    psd = psd / (psd.sum() + eps)
    ent = -(psd * np.log(psd + eps)).sum()
    return float(ent / np.log(len(psd) + eps))

def dominant_freq_ratio_1d(x, eps=1e-8):
    fft_mag = np.abs(np.fft.rfft(x))
    power = fft_mag ** 2
    return float(power.max() / (power.sum() + eps))

def low_high_ratio_1d(x, eps=1e-8):
    power = np.abs(np.fft.rfft(x)) ** 2
    if len(power) < 4:
        return 1.0
    mid = max(1, len(power) // 3)
    low = power[:mid].sum()
    high = power[mid:].sum()
    return float(low / (high + eps))

def safe_corr(a, b):
    a, b = np.asarray(a), np.asarray(b)
    if a.std() < 1e-8 or b.std() < 1e-8:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])

def extract_patch_physics_np(sample, patch_size=8):
    T, C = sample.shape
    N = T // patch_size
    feats = []
    for i in range(N):
        patch = sample[i * patch_size : (i + 1) * patch_size]
        body_acc  = patch[:, 0:3]
        body_gyro = patch[:, 3:6]
        total_acc = patch[:, 6:9]

        acc_mag = np.linalg.norm(body_acc, axis=-1)
        gyr_mag = np.linalg.norm(body_gyro, axis=-1)
        tot_mag = np.linalg.norm(total_acc, axis=-1)
        jerk = np.diff(acc_mag, prepend=acc_mag[0])

        acc_energy  = float(np.mean(acc_mag ** 2))
        gyr_energy  = float(np.mean(gyr_mag ** 2))
        jerk_energy = float(np.mean(jerk ** 2))

        spec_ent = 0.5 * (
            spectral_entropy_1d(acc_mag) + spectral_entropy_1d(gyr_mag)
        )
        dom_ratio = 0.5 * (
            dominant_freq_ratio_1d(acc_mag) + dominant_freq_ratio_1d(gyr_mag)
        )
        low_high = 0.5 * (
            low_high_ratio_1d(acc_mag) + low_high_ratio_1d(gyr_mag)
        )

        gravity_proxy = total_acc - body_acc
        gravity_mag = np.linalg.norm(gravity_proxy, axis=-1)
        gravity_body_consistency = safe_corr(gravity_mag, tot_mag)
        acc_gyro_coupling = safe_corr(acc_mag, gyr_mag)

        axis_corrs = [
            safe_corr(body_acc[:, 0], body_acc[:, 1]),
            safe_corr(body_acc[:, 1], body_acc[:, 2]),
            safe_corr(body_acc[:, 0], body_acc[:, 2]),
        ]
        axis_correlation = float(np.mean(axis_corrs))

        feats.append([
            acc_energy, gyr_energy, jerk_energy,
            spec_ent, dom_ratio, low_high,
            gravity_body_consistency, acc_gyro_coupling, axis_correlation,
        ])
    return np.asarray(feats, dtype=np.float32)

def build_patch_physics_table(X, patch_size=8):
    all_feats = [extract_patch_physics_np(x, patch_size) for x in X]
    all_feats = np.stack(all_feats).astype(np.float32)
    mu = all_feats.reshape(-1, all_feats.shape[-1]).mean(axis=0, keepdims=True)
    std = all_feats.reshape(-1, all_feats.shape[-1]).std(axis=0, keepdims=True) + 1e-8
    all_feats = (all_feats - mu[None, :, :]) / std[None, :, :]
    return all_feats, mu, std

def apply_patch_physics_norm(X, phys_mu, phys_std, patch_size=8):
    feats = np.stack([extract_patch_physics_np(x, patch_size) for x in X]).astype(np.float32)
    feats = (feats - phys_mu[None, :, :]) / phys_std[None, :, :]
    return feats

print("Building patch-level physics descriptors...")
phys_train, phys_mu, phys_std = build_patch_physics_table(X_train, PATCH_SIZE)
phys_val  = apply_patch_physics_norm(X_val,  phys_mu, phys_std, PATCH_SIZE)
phys_test = apply_patch_physics_norm(X_test, phys_mu, phys_std, PATCH_SIZE)

PHYS_DIM = phys_train.shape[-1]
print(f"Physics train : {phys_train.shape}")
print(f"Physics val   : {phys_val.shape}")
print(f"Physics test  : {phys_test.shape}")
print(f"PHYS_DIM      : {PHYS_DIM}")

# ============================================================
# 4) Dataset & DataLoader
# ============================================================
class MHealthDataset(Dataset):
    def __init__(self, X, y, phys):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
        self.phys = torch.FloatTensor(phys)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.phys[idx], self.y[idx]

train_ds = MHealthDataset(X_train, y_train, phys_train)
val_ds   = MHealthDataset(X_val,   y_val,   phys_val)
test_ds  = MHealthDataset(X_test,  y_test,  phys_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train loader: {len(train_loader)} batches")
print(f"Val   loader: {len(val_loader)} batches")
print(f"Test  loader: {len(test_loader)} batches")

# ============================================================
# 5) Time / Frequency Patch Embedding
# ============================================================
class TimePatchEmbed(nn.Module):
    def __init__(self, num_channels, patch_size, d_model):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Linear(patch_size * num_channels, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        B, T, C = x.shape
        N = T // self.patch_size
        x = x[:, :N * self.patch_size, :]
        x = x.reshape(B, N, self.patch_size * C)
        x = self.proj(x)
        return self.norm(x)

class SpectralFilterbankEmbed(nn.Module):
    def __init__(self, num_channels, patch_size, num_bands, d_model):
        super().__init__()
        self.patch_size = patch_size
        self.num_channels = num_channels
        self.num_bins = patch_size // 2 + 1
        self.num_bands = num_bands
        self.filterbank = nn.Parameter(
            torch.randn(num_channels, self.num_bins, num_bands) * 0.02
        )
        self.proj = nn.Linear(num_channels * num_bands, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        B, T, C = x.shape
        N = T // self.patch_size
        x = x[:, :N * self.patch_size, :]
        x = x.reshape(B, N, self.patch_size, C)
        xf = torch.fft.rfft(x, dim=2)
        xf = torch.abs(xf)
        xf = xf.permute(0, 1, 3, 2)
        bands = torch.einsum('bncf,cfk->bnck', xf, self.filterbank)
        bands = bands.reshape(B, N, C * self.num_bands)
        z = self.proj(bands)
        return self.norm(z)

# ============================================================
# 6) Transformer Blocks
# ============================================================
class TransformerBlock(nn.Module):
    def __init__(self, d_model, nhead, d_ff, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=nhead,
            dropout=dropout, batch_first=True
        )
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x1 = self.norm1(x)
        z, _ = self.attn(x1, x1, x1)
        x = x + z
        x = x + self.ffn(self.norm2(x))
        return x

class BranchEncoder(nn.Module):
    def __init__(self, d_model, nhead, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, nhead, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        return self.norm(x)

# ============================================================
# 7) Patch-Level Physics Gate (Ours)
# ============================================================
class PatchPhysicsGate(nn.Module):
    def __init__(self, in_dim, hidden=64, temp_init=2.0):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, 1),
        )
        self.last_alpha_mean = 0.0

    def forward(self, z_phys_patch):
        logits = self.mlp(z_phys_patch)
        alpha = torch.sigmoid(logits)
        self.last_alpha_mean = alpha.mean().item()
        return alpha

# ============================================================
# 7-A) Ablation: Fixed Average Gate (alpha = 0.5)
# ============================================================
class FixedAverageGate(nn.Module):
    """Ablation baseline: always alpha=0.5 (equal fusion)."""
    def __init__(self, in_dim, hidden=64, temp_init=2.0):
        super().__init__()
        self.last_alpha_mean = 0.5

    def forward(self, z_phys_patch):
        B, N, _ = z_phys_patch.shape
        alpha = torch.full((B, N, 1), 0.5, device=z_phys_patch.device)
        return alpha

# ============================================================
# 7-B) Ablation: Self-Attention Fusion Gate (no physics)
# ============================================================
class SelfAttentionFusionGate(nn.Module):
    """
    Ablation: data-driven gate using cross-attention between
    time and freq branch outputs. Does NOT use physics descriptors.
    """
    def __init__(self, d_model, nhead=2):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=nhead,
            dropout=0.1, batch_first=True
        )
        self.gate_proj = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, 1),
        )
        self.last_alpha_mean = 0.5

    def forward(self, ht, hf):
        # ht, hf: (B, N, D) — time/freq branch outputs
        attn_out, _ = self.cross_attn(ht, hf, hf)
        logits = self.gate_proj(attn_out)           # (B, N, 1)
        alpha = torch.sigmoid(logits)
        self.last_alpha_mean = alpha.mean().item()
        return alpha

# ============================================================
# 8) Full Model (Physics-Guided Gate)
# ============================================================
class PhysicsGuidedPatchDualTransformer(nn.Module):
    def __init__(
        self, num_channels, patch_size, num_patches, d_model, nhead,
        num_layers, d_ff, dropout, num_classes, phys_dim,
        gate_hidden=64, temp_init=2.0, num_freq_bands=8,
    ):
        super().__init__()
        self.num_patches = num_patches
        self.d_model = d_model

        self.time_embed = TimePatchEmbed(num_channels, patch_size, d_model)
        self.freq_embed = SpectralFilterbankEmbed(
            num_channels, patch_size, num_freq_bands, d_model
        )

        self.time_pos = nn.Parameter(torch.zeros(1, num_patches, d_model))
        self.freq_pos = nn.Parameter(torch.zeros(1, num_patches, d_model))
        nn.init.trunc_normal_(self.time_pos, std=0.02)
        nn.init.trunc_normal_(self.freq_pos, std=0.02)

        self.time_encoder = BranchEncoder(d_model, nhead, d_ff, num_layers, dropout)
        self.freq_encoder = BranchEncoder(d_model, nhead, d_ff, num_layers, dropout)

        self.gate = PatchPhysicsGate(
            in_dim=phys_dim, hidden=gate_hidden, temp_init=temp_init
        )

        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )

    def forward(self, x, z_phys_patch):
        ht = self.time_embed(x) + self.time_pos
        hf = self.freq_embed(x) + self.freq_pos

        ht = self.time_encoder(ht)
        hf = self.freq_encoder(hf)

        alpha = self.gate(z_phys_patch)
        h_patch = alpha * ht + (1.0 - alpha) * hf

        hp_pool = h_patch.mean(dim=1)
        logits = self.classifier(hp_pool)
        return logits, alpha.squeeze(-1), ht, hf, h_patch

    def get_alpha_mean(self):
        return self.gate.last_alpha_mean

# ============================================================
# 8-A) Ablation Model: Fixed Average
# ============================================================
class FixedAverageModel(PhysicsGuidedPatchDualTransformer):
    """Same architecture but gate always returns 0.5."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.gate = FixedAverageGate(in_dim=kwargs['phys_dim'])

    def get_alpha_mean(self):
        return 0.5

# ============================================================
# 8-B) Ablation Model: Self-Attention Fusion
# ============================================================
class SelfAttentionFusionModel(PhysicsGuidedPatchDualTransformer):
    """
    Replace physics gate with self-attention fusion.
    Gate uses time/freq branch outputs instead of physics descriptors.
    """
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.sa_gate = SelfAttentionFusionGate(
            d_model=kwargs['d_model'], nhead=2
        )
        # remove unused physics gate
        del self.gate

    def forward(self, x, z_phys_patch):
        ht = self.time_embed(x) + self.time_pos
        hf = self.freq_embed(x) + self.freq_pos

        ht = self.time_encoder(ht)
        hf = self.freq_encoder(hf)

        alpha = self.sa_gate(ht, hf)                # (B,N,1)
        h_patch = alpha * ht + (1.0 - alpha) * hf

        hp_pool = h_patch.mean(dim=1)
        logits = self.classifier(hp_pool)
        return logits, alpha.squeeze(-1), ht, hf, h_patch

    def get_alpha_mean(self):
        return self.sa_gate.last_alpha_mean

# ============================================================
# 9) Train / Eval — Validation 기반 모델 선택
# ============================================================
def train_model(model, train_loader, val_loader, epochs, lr, warmup_ep, label):
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)

    def lr_lambda(ep):
        if ep < warmup_ep:
            return (ep + 1) / warmup_ep
        p = (ep - warmup_ep) / max(1, epochs - warmup_ep)
        return 0.5 * (1.0 + np.cos(np.pi * p))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_val_acc = 0.0
    best_state = None
    t0 = time.time()

    for ep in range(1, epochs + 1):
        # ── Train ──
        model.train()
        tr_loss_sum, tr_corr, tr_n = 0.0, 0, 0

        for X_b, phys_b, y_b in train_loader:
            X_b, phys_b, y_b = X_b.to(device), phys_b.to(device), y_b.to(device)
            optimizer.zero_grad()

            logits, alpha, _, _, _ = model(X_b, phys_b)
            ce_loss = criterion(logits, y_b)

            if ep <= 10:
                balance_loss = (alpha.mean() - 0.5).abs()
                loss = ce_loss + 0.01 * balance_loss
            else:
                loss = ce_loss

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            tr_loss_sum += ce_loss.item() * len(y_b)
            tr_corr += (logits.argmax(dim=1) == y_b).sum().item()
            tr_n += len(y_b)

        scheduler.step()

        # ── Validation ──
        model.eval()
        val_corr, val_n = 0, 0
        with torch.no_grad():
            for X_b, phys_b, y_b in val_loader:
                X_b, phys_b, y_b = X_b.to(device), phys_b.to(device), y_b.to(device)
                logits, _, _, _, _ = model(X_b, phys_b)
                val_corr += (logits.argmax(dim=1) == y_b).sum().item()
                val_n += len(y_b)

        val_acc = val_corr / val_n

        # Best model 선택은 validation accuracy 기준
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        if ep == 1 or ep % 10 == 0:
            print(
                f"[{label}] ep{ep:3d} | "
                f"loss={tr_loss_sum / max(tr_n, 1):.4f} "
                f"tr_acc={tr_corr / max(tr_n, 1):.4f} "
                f"val_acc={val_acc:.4f} best_val={best_val_acc:.4f} "
                f"alpha={model.get_alpha_mean():.3f}"
            )

    print(f"[{label}] Done. Best Val Acc={best_val_acc:.4f} ({time.time() - t0:.0f}s)")
    return best_val_acc, best_state

@torch.no_grad()
def evaluate_model(model, loader, label):
    model.eval()
    all_preds, all_labels, all_alpha = [], [], []
    alpha_by_class = {i: [] for i in range(NUM_CLASSES)}

    for X_b, phys_b, y_b in loader:
        X_b, phys_b = X_b.to(device), phys_b.to(device)
        logits, alpha, _, _, _ = model(X_b, phys_b)
        preds = logits.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds.tolist())
        all_labels.extend(y_b.numpy().tolist())
        all_alpha.extend(alpha.mean(axis=1).cpu().numpy().tolist())

        for a, y in zip(alpha.mean(axis=1).cpu().numpy(), y_b.numpy()):
            alpha_by_class[int(y)].append(float(a))

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_alpha  = np.array(all_alpha)

    print(f"\n{'=' * 70}")
    print(label)
    print(f"{'=' * 70}")
    print(classification_report(
        all_labels, all_preds,
        target_names=ACTIVITY_NAMES, digits=4, zero_division=0
    ))

    print("Routing statistics:")
    print(f"  mean alpha (time-branch weight): {all_alpha.mean():.4f}")
    print(f"  mean freq weight               : {(1 - all_alpha).mean():.4f}")
    for c in range(NUM_CLASSES):
        arr = np.array(alpha_by_class[c]) if alpha_by_class[c] else np.array([0.0])
        print(f"  {ACTIVITY_NAMES[c]:20s} alpha={arr.mean():.4f}  freq={(1 - arr).mean():.4f}")

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    acc = (all_labels == all_preds).mean()
    return acc, macro_f1, all_preds, all_labels, all_alpha

# ============================================================
# 10) Routing Ablation Study
# ============================================================
model_configs = {
    "Fixed Average (0.5:0.5)": FixedAverageModel,
    "Self-Attention Fusion":   SelfAttentionFusionModel,
    "Physics-Guided Gate (Ours)": PhysicsGuidedPatchDualTransformer,
}

common_kwargs = dict(
    num_channels=NUM_CHANNELS,
    patch_size=PATCH_SIZE,
    num_patches=N_PATCHES,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT,
    num_classes=NUM_CLASSES,
    phys_dim=PHYS_DIM,
    gate_hidden=GATE_HIDDEN,
    num_freq_bands=NUM_FREQ_BANDS,
)

ablation_results = {}

for name, ModelClass in model_configs.items():
    print(f"\n{'='*70}")
    print(f"  Ablation: {name}")
    print(f"{'='*70}")

    # Reset seed before each run for fair comparison
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    model = ModelClass(**common_kwargs).to(device)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable params: {n_params:,}")

    best_val_acc, best_state = train_model(
        model, train_loader, val_loader,
        epochs=EPOCHS,
        lr=LR,
        warmup_ep=WARMUP_EP,
        label=name
    )

    model.load_state_dict(best_state)

    # Evaluate on test set
    acc, macro_f1, preds, labels, alphas = evaluate_model(
        model, test_loader, f"{name} — Test Set"
    )

    ablation_results[name] = {"acc": acc, "f1": macro_f1, "val_acc": best_val_acc}
    print(f"  => Val Acc={best_val_acc:.4f}, Test Acc={acc:.4f}, Macro-F1={macro_f1:.4f}")

# ============================================================
# 11) Ablation Summary Table
# ============================================================
print(f"\n{'='*70}")
print("TABLE: Routing Ablation Study (mHealth)")
print(f"{'='*70}")
print(f"{'Fusion Strategy':<30s}  {'Val Acc':>7s}  {'Test Acc':>8s}  {'Macro-F1':>8s}")
print("-" * 60)
for name in model_configs:
    r = ablation_results[name]
    bold = " **" if name == "Physics-Guided Gate (Ours)" else ""
    print(f"{name:<30s}  {r['val_acc']*100:6.2f}   {r['acc']*100:7.2f}   {r['f1']*100:7.2f}{bold}")
print("-" * 60)
print("Done.")


Device: cpu
NUM_CLASSES : 12
NUM_CHANNELS: 9
WINDOW_SIZE : 128 (50Hz x 2.56s)
TEST_SIZE   : 0.2
VAL_SIZE    : 0.1

Loading mHealth...
  mHealth_subject1.log: 535 windows
  mHealth_subject2.log: 542 windows
  mHealth_subject3.log: 538 windows
  mHealth_subject4.log: 537 windows
  mHealth_subject5.log: 517 windows
  mHealth_subject6.log: 489 windows
  mHealth_subject7.log: 517 windows
  mHealth_subject8.log: 505 windows
  mHealth_subject9.log: 522 windows
  mHealth_subject10.log: 511 windows

Total: (5213, 128, 9), labels: (5213,)
Class distribution: {np.int64(0): np.int64(470), np.int64(1): np.int64(470), np.int64(2): np.int64(470), np.int64(3): np.int64(470), np.int64(4): np.int64(463), np.int64(5): np.int64(428), np.int64(6): np.int64(443), np.int64(7): np.int64(443), np.int64(8): np.int64(469), np.int64(9): np.int64(470), np.int64(10): np.int64(470), np.int64(11): np.int64(147)}

Train : (3753, 128, 9)  (3753 samples)
Val   : (417, 128, 9)  (417 samples)
Test  : (1043, 128, 9)  (1043